In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load both datasets
restaurant_df = pd.read_csv("restaurant_with_hygiene_score.csv")
menu_df       = pd.read_csv("menu_data.csv")

# Merge on BusinessName (keep all restaurants even if menu is missing)
merged_df = pd.merge(menu_df, restaurant_df, on="BusinessName", how="left")

print(merged_df.shape)
print(merged_df.head(10))

(3453, 13)
   BusinessName      Cuisine                DishName  DishRating  DishPrice  \
0  Restaurant_1      British         Beef Wellington         3.3       4.54   
1  Restaurant_1      British         Victoria Sponge         2.6      16.86   
2  Restaurant_1      British            Fish & Chips         3.1      10.90   
3  Restaurant_1      British          Beans on Toast         2.9       4.36   
4  Restaurant_1      British   Sticky Toffee Pudding         3.2      10.58   
5  Restaurant_1      British          Shepherd's Pie         2.7      21.69   
6  Restaurant_1      British  Full English Breakfast         3.7      16.09   
7  Restaurant_1      British              Scotch Egg         3.0      11.83   
8  Restaurant_2  South Asian            Palak Paneer         1.8       4.14   
9  Restaurant_2  South Asian             Dal Makhani         1.3      20.32   

           BusinessType RatingValue InspectionDate    Address  PostCode  \
0  Restaurant/Cafe/Shop        Pass     2024

In [ ]:
# Pivot: Business (rows) x Dish (columns), values = DishRating
pivot = merged_df.pivot_table(
    index   = "BusinessName",
    columns = "DishName",
    values  = "DishRating",
    aggfunc = "mean"       # average if a dish appears twice for same business
)

print(pivot.shape)   # e.g. (500, 60) — 500 restaurants, ~60 unique dishes
print(pivot.head())

(530, 80)
DishName        Avocado Toast  BBQ Wings  Baklava  Beans on Toast  \
BusinessName                                                        
Restaurant_1              NaN        NaN      NaN             2.9   
Restaurant_10             NaN        NaN      NaN             NaN   
Restaurant_100            NaN        NaN      NaN             NaN   
Restaurant_101            4.8        NaN      NaN             NaN   
Restaurant_102            NaN        NaN      NaN             NaN   

DishName        Beef Burger  Beef Tacos  Beef Wellington  Blueberry Muffin  \
BusinessName                                                                 
Restaurant_1            NaN         NaN              3.3               NaN   
Restaurant_10           NaN         NaN              NaN               NaN   
Restaurant_100          NaN         NaN              NaN               NaN   
Restaurant_101          NaN         NaN              NaN               5.0   
Restaurant_102          1.6         Na

In [ ]:
#  Mean-center each restaurant's ratings first
# This removes bias from restaurants that rate everything high/low
pivot_normalized = pivot.apply(
    lambda row: row - row.mean(), axis=1
).fillna(0)

In [ ]:
# Transpose so dishes are rows (we want DISH-to-DISH similarity)
dish_matrix = pivot_normalized.T    # shape: (n_dishes, n_restaurants)

# Compute cosine similarity between every pair of dishes
cos_sim = cosine_similarity(dish_matrix)

# Wrap in a labeled DataFrame
dish_sim_df = pd.DataFrame(
    cos_sim,
    index   = pivot.columns,
    columns = pivot.columns
)

print(dish_sim_df.shape)            # e.g. (60, 60)
print(dish_sim_df["Beef Burger"].sort_values(ascending=False).head())

(80, 80)
DishName
Beef Burger    1.000000
Fish Fillet    0.085809
BBQ Wings      0.040335
Coleslaw       0.030437
Hot Dog        0.016356
Name: Beef Burger, dtype: float64


In [ ]:
# --- Recommend dishes similar to a given dish ---
def recommend_similar_dishes(dish_name, n=5):
    if dish_name not in dish_sim_df:
        return f"'{dish_name}' not found in menu."
    scores = dish_sim_df[dish_name].drop(dish_name)
    return scores.nlargest(n).reset_index()

# --- Recommend restaurants for a given dish ---
def recommend_restaurants(dish_name, n=5):
    subset = merged_df[merged_df["DishName"] == dish_name]
    return (subset
            .nlargest(n, "DishRating")[["BusinessName", "DishRating", "Province"]]
            .reset_index(drop=True))

# --- Recommend dishes for a given restaurant ---
def recommend_for_restaurant(business_name, n=5):
    if business_name not in pivot_normalized.index:
        return "Restaurant not found."
    # Get this restaurant's rated dishes
    rated = pivot.loc[business_name].dropna().index.tolist()
    # Aggregate similarity scores from all rated dishes
    scores = dish_sim_df[rated].mean(axis=1)
    # Exclude dishes already on the menu
    scores = scores.drop(labels=rated, errors="ignore")
    return scores.nlargest(n).reset_index()

In [ ]:
print(recommend_similar_dishes("Chicken Tikka Masala", n=5))

print(recommend_restaurants("Chicken Tikka Masala", n=5))


        DishName  Chicken Tikka Masala
0    Garlic Naan              0.110935
1    Mango Lassi              0.088459
2  Avocado Toast              0.000000
3      BBQ Wings              0.000000
4        Baklava              0.000000
     BusinessName  DishRating Province
0  Restaurant_145         5.0   Punjab
1  Restaurant_413         5.0   Punjab
2  Restaurant_288         4.7   Punjab
3  Restaurant_371         4.6   Punjab
4   Restaurant_77         4.5   Punjab
